# General Overview

Pytorch offers some niceties for model training particularly with separating the model definition and optimization pieces.
This makes it easy to update different models and try out various optimization methods wihtout having to
deal with the complex and error-prone optimization code.

The high level steps to training a model on data is as follows:

1. Raw Data (`ratings.csv`)
2. Wrap data with `Dataset` class: index-able data
3. Wrap data with `Dataloader` class: organizes data for training (batching, shuffling, etc)
4. Loop through the `Dataloader` to get a batch of data.
5. The batch (typically a `tensor` of values) is given to the model.
6. The model calculates gradient from the batch of data.
7. Take a gradient step with `optimizer` like `SGD` or `Adam` variants.
8. Loop till training is done.

https://pytorch.org/tutorials/beginner/basics/intro.html

## Our experiments

Our experiments are slightly different since we are now dealing with multiple models.
However the main ideas are still the same.
The main restriction now is that when we grab a `batch` of data from the `Dataloader`, all the data must be from the same user.

This means that we need custom `Dataset`'s to ensure this happens in a reasonable way.

1. Read in data
2. Train, validation, test split
3. Convert to `Dataloader` (OAAT, RS, URS, etc)
4. Initalize models for each user
5. Create graph
6. For each of $n$ epochs
    1. For each batch in `Dataloader` (user_id, item_id), (target)
       1. Calculate loss
       2. User model's gradient step
       3. Share gradient with neighbors
    3. Check validation error for early stopping.
    4. Log metrics
7. Calculate and log test error

### 1 Read in Data
Read in csv files into pandas `DataFrames`.

```python
def read_in_data(data_name) -> Tuple[pd.DataFrame, pd.DataFrame]:
    if data_name == "HM":
        train_df = pd.read_csv(DATA_DIR / "hm_train_df.csv")
        val_df = pd.read_csv(DATA_DIR / "hm_val_df.csv")
        train_df.columns = ["user_id", "item_id", "bought"]
        val_df.columns = ["user_id", "item_id", "bought"]
    elif data_name == "HM_4000":
        train_df = pd.read_csv(DATA_DIR / "hm_4000_item_subset_train.csv")
        val_df = pd.read_csv(DATA_DIR / "hm_4000_item_subset_test.csv")
    elif data_name == "HM_2881":
        train_df = pd.read_csv(DATA_DIR / "hm_2881_item_subset_train.csv")
        val_df = pd.read_csv(DATA_DIR / "hm_2881_item_subset_test.csv")
    elif data_name == "HM_SUBSET":
        train_df = pd.read_csv(DATA_DIR / "hm_subset_train.csv")
        val_df = pd.read_csv(DATA_DIR / "hm_subset_test.csv")
    elif data_name == "ML100K":
        train_df = pd.read_csv(DATA_DIR / "ml100k_train.csv")
        val_df = pd.read_csv(DATA_DIR / "ml100k_test.csv")
    elif data_name == "ML100K_100":
        train_df = pd.read_csv(DATA_DIR / "ml100k_100user_train.csv")
        val_df = pd.read_csv(DATA_DIR / "ml100k_100user_test.csv")
    else:
        raise NotImplementedError
    return train_df, val_df

```

### 3 Convert to Dataloader
Convert the pandas `DataFrames` into torch `Tensors` and wrap them with a `Dataset` and then a `DataLoader`.
We created some custom `Dataset` classes to make the training happen in a specific way.

```python
def create_dataloader(df, dl_type, batch_size=1, p=None) -> DataLoader:
    assert dl_type in DL_TYPES

    X, y = df.iloc[:, :2].to_numpy(), df.iloc[:, 2].to_numpy()
    if dl_type == "oaat":
        return DataLoader(
            TensorDataset(torch.tensor(X), torch.tensor(y).unsqueeze(-1)),
            batch_size=1,
            shuffle=True,
        )
    elif dl_type == "aaat":
        return DataLoader(UserStratifiedDataset(X, y), shuffle=True)
    elif dl_type == "rs":
        return DataLoader(
            UserRandomSamplingDataset(X, y, batch_size=batch_size),
            batch_size=1,
            shuffle=True,
        )
    elif dl_type == "centralized":
        return DataLoader(
            TensorDataset(torch.tensor(X), torch.tensor(y).unsqueeze(-1)),
            batch_size=batch_size,
            shuffle=True,
        )
    elif dl_type == "urs":
        return DataLoader(
            UserStratifiedDataset(X, y, batch_size=batch_size),
            batch_size=1,
            shuffle=True,
        )
    elif dl_type == "userprop":
        assert p is not None
        return DataLoader(
            UserProportionDataset(X, y, p),
            batch_size=1,
            shuffle=True,
        )
    else:
        raise NotImplementedError
```
For example, the `rs` train type uses the `UserRandomSamplingDataset`.
Given a user id, this dataset will return $n$ items randomly sampled from that user's interactions.
```python
class UserRandomSamplingDataset(Dataset):
    def __init__(self, x: np.ndarray, y: np.ndarray, batch_size: int = 10):
        self.x = torch.tensor(x)
        self.y = torch.tensor(y).unsqueeze(-1)
        self.batch_size = batch_size

    def __len__(self):
        # return len(self.x[:, 0].unique())
        return self.x.shape[0]

    def __getitem__(self, idx):
        user = self.x[idx, 0]
        user_idx = self.x[:, 0] == user
        x = self.x[user_idx, :]
        y = self.y[user_idx]
        batch_size = min(x.shape[0], self.batch_size)
        selected_indices = torch.randint(x.shape[0], size=(batch_size,))
        return x[selected_indices, :], y[selected_indices]

```

### 4 Model Initialization
The `User` custom class just groups a user's model, optimizer, and name for ease of use.
```python
    users = {}
    for i in tqdm(range(n_users)):
        if model == "umf":
            user_model = UMF(n_items, n_factors=n_factors, sparse=sparse)
        elif model == "gmf":
            layer_size = 5
            user_model = GeneralizedMF(n_items, n_factors=n_factors, layer_size=layer_size)
            mlflow.log_param("layer_size", layer_size)
        elif model == "gmf_shared":
            layer_size = 5
            user_model = GeneralizedMFSharedItemLayer(n_items, n_factors, layer_size=layer_size)
            mlflow.log_param("layer_size", layer_size)
        else:
            user_model = MF(n_users, n_items, n_factors=n_factors, sparse=sparse)
        users[i] = User(
            id=i,
            model=user_model,
            optimizer=SGD(user_model.parameters(), lr=lr, weight_decay=weight_decay, momentum=mom),
            model_name=model,
        )
```

### What do Models look like?

Models are relatively simple looking.
You really only need to define `__init__` and `forward`
* `__init__`: the initialization method `__init__` which defines the model pieces
* `forward`: the "forward" pass of the model. It takes inputs and returns outputs. This describes how data passes through those model pieces.

#### Matrix Factorization
```python
class MF(nn.Module):
    def __init__(self, n_users, n_items, n_factors=30, sparse=False):
        super().__init__()
        self.user_factors = nn.Embedding(n_users, n_factors, sparse=sparse)
        self.item_factors = nn.Embedding(n_items, n_factors, sparse=sparse)
        self.user_bias = nn.Embedding(n_users, 1, sparse=sparse)
        self.item_bias = nn.Embedding(n_items, 1, sparse=sparse)

    def forward(self, users, items):
        preds = (self.user_factors(users) * self.item_factors(items)).sum(dim=1, keepdims=True)
        preds += self.user_bias(users)
        preds += self.item_bias(items)
        return preds
```

#### Generalized Matrix Factorization
```python
class GeneralizedMF(nn.Module):
    def __init__(self, n_items, n_factors=30, layer_size=10):
        super().__init__()

        self.user_factors = nn.Embedding(1, embedding_dim=n_factors).float()
        self.item_factors = nn.Embedding(n_items, embedding_dim=n_factors).float()
        self.user_bias = nn.Embedding(1, embedding_dim=1).float()
        self.item_bias = nn.Embedding(n_items, embedding_dim=1).float()

        self.user_layers = nn.Linear(n_factors, layer_size)
        self.item_layer_dict = nn.ModuleDict()
        for i in range(n_items):
            self.item_layer_dict["item{0}".format(i)] = nn.Linear(n_factors, layer_size)

    def forward(self, users, items):
        user = tensor(0)
        u_e = self.user_factors(user)
        i_e = self.item_factors(items)
        u_e = self.user_layers(u_e)
        i_e = self.item_layer_dict["item{0}".format(int(items))](i_e)

        u_b = self.user_bias(user)
        i_b = self.item_bias(items)
        x = (u_e * i_e).sum(1) + u_b + i_b
        return x
```

## 6.A Training Loop
```python
def decentralized_train_loop(user_models, train_loader, graph, progress_bar=True):
    loss_fn = nn.MSELoss(reduction="mean")
    for i, user in user_models.items():
        user.model.train()
    losses = np.empty(len(train_loader))
    total_n_obs = 0
    total_sum_loss = 0
    avg_loss = 0
    tbar = tqdm(train_loader) if progress_bar else train_loader
    for idx, (inputs, target) in enumerate(tbar):
        if inputs.ndim == 3:
            inputs = inputs.squeeze(0)
            target = target.squeeze(0)
        n_obs = inputs.shape[0]
        user_id = int(inputs[0, 0])
        user = user_models[user_id]
        optimizer = user.optimizer
        optimizer.zero_grad()
        score = user.model(inputs[:, 0], inputs[:, 1])
        loss = loss_fn(score, target.float())
        loss.backward()  # Calculate Gradients

        if inputs[:,1].numel() == 1:
            share_gradient(user, user_models, graph, item_id=inputs[:,1].item())
        else:
            share_gradient(user, user_models, graph)
        optimizer.step()  # Current user's update

        total_n_obs += n_obs
        total_sum_loss += loss.detach().numpy() * n_obs
        losses[idx] = loss.detach().numpy()
        avg_loss = avg_loss * (idx / (idx + 1)) + loss.detach().numpy() / (idx + 1) / n_obs
        if idx % 1000 == 0:
            if progress_bar:
                tbar.set_description(
                    f"Average Training Loss: {np.sqrt(avg_loss):.04f} | Loss: {loss.detach():.04f}"
                )
            if np.isnan(avg_loss):
                raise ValueError
    return avg_loss
```

## 6.A.C Gradient Sharing
```python
def share_gradient(user, users, graph, item_id=None):
    user_neighbors = graph.adj[user.id]
    item_parameter_names = ["item_factors.weight", "item_bias.weight"]
    if user.model_name == "gmf":
        item_parameter_names.append(f"item_layer_dict.item{item_id}.weight")
        item_parameter_names.append(f"item_layer_dict.item{item_id}.bias")
    elif user.model_name == "gmf_shared":
        item_parameter_names.append("item_layers.weight")
        item_parameter_names.append("item_layers.bias")

    user_gradients = {name: user.model.get_parameter(name).grad for name in item_parameter_names}

    for neighbor_id in user_neighbors:
        neighbor = users[neighbor_id]
        neighbor_model = neighbor.model
        neighbor_optimizer = neighbor.optimizer
        neighbor_model.zero_grad() # required - diverges without this line
        for name in item_parameter_names:
            param = neighbor_model.get_parameter(name)
            param.grad = user_gradients[name].clone()
            
        neighbor_optimizer.step()  # Neighbors' update
```